# Notebook 04 — Enhanced 3D ResUNet Model Architecture

This notebook defines, builds, validates, and persistently saves the Enhanced 3D ResUNet multi-task model for:

1. Binary rib-fracture segmentation
2. Five-class fracture patch classification

Input: `64 × 64 × 64 × 1` CT patch.

Classification classes:

| Class | Meaning |
|---:|---|
| 0 | Background / negative patch |
| 1 | Displaced fracture |
| 2 | Non-displaced fracture |
| 3 | Buckle fracture |
| 4 | Segmental fracture |

### Notebook separation
- Notebook 03: patch extraction and labels
- Notebook 04: architecture, validation, and persistent model artifacts
- Notebook 05: training
- Notebook 06: final evaluation

Notebook 04 does **not** train the model. Successful execution establishes architectural and computational validity.

### Persistent output
The notebook creates:

```text
/kaggle/working/model_artifacts/
├── enhanced_3d_resunet_multitask.keras
├── model_config.json
└── dataset-metadata.json
```

The directory is intended to be published as the persistent Kaggle Dataset:

`waboke/ribfrac-model-artifacts`

In [ ]:
# CELL 1 — IMPORTS AND REPRODUCIBILITY

import os
import json
import random
from pathlib import Path

import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

SEED = 42

os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print("TensorFlow:", tf.__version__)
print("Seed:", SEED)

In [ ]:
# CELL 2 — GPU CONFIGURATION

gpus = tf.config.list_physical_devices("GPU")
print("Number of GPUs:", len(gpus))

if gpus:
    for gpu in gpus:
        print(gpu)
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU memory growth enabled.")
    except RuntimeError as error:
        print("GPU configuration warning:", error)
else:
    print("WARNING: No GPU detected.")

## 1. Persistent input paths

The code supports both the published patch dataset and the current development path.

In [ ]:
# CELL 3 — LOCATE PATCH DATASET

PATCH_DATASET_CANDIDATES = [
    Path("/kaggle/input/datasets/waboke/ribfrac-patch-dataset"),
    Path("/kaggle/input/ribfrac-patch-dataset"),
    Path("/kaggle/input/datasets/waboke/ribfrac-processed-dev"),
    Path("/kaggle/working/patch_dataset"),
]

PATCH_DATASET_ROOT = next(
    (p for p in PATCH_DATASET_CANDIDATES if p.exists()),
    None
)

if PATCH_DATASET_ROOT is None:
    raise FileNotFoundError(
        "Could not locate the RibFrac patch dataset. Checked:\n"
        + "\n".join(str(p) for p in PATCH_DATASET_CANDIDATES)
    )

if (PATCH_DATASET_ROOT / "patch_dataset").exists():
    PATCH_ROOT = PATCH_DATASET_ROOT / "patch_dataset"
else:
    PATCH_ROOT = PATCH_DATASET_ROOT

RIBFRAC128_ROOT = PATCH_ROOT / "RibFrac128"

X_PATH = RIBFRAC128_ROOT / "X.npy"
Y_SEG_PATH = RIBFRAC128_ROOT / "Y_seg.npy"
Y_CLASS_PATH = RIBFRAC128_ROOT / "Y_class.npy"

print("Patch dataset root:", PATCH_DATASET_ROOT)
print("Patch root:", PATCH_ROOT)
print("RibFrac128 root:", RIBFRAC128_ROOT)

In [ ]:
# CELL 4 — VERIFY NOTEBOOK 03 OUTPUTS

required_patch_files = [X_PATH, Y_SEG_PATH, Y_CLASS_PATH]

for path in required_patch_files:
    assert path.exists(), f"Missing required file: {path}"

print("Notebook 03 output validation: PASS")
for path in required_patch_files:
    print(" ", path)

In [ ]:
# CELL 5 — LOAD DEVELOPMENT PATCH ARRAYS

X_dev = np.load(X_PATH, mmap_mode="r")
Y_seg_dev = np.load(Y_SEG_PATH, mmap_mode="r")
Y_class_dev = np.load(Y_CLASS_PATH, mmap_mode="r")

print("X shape:", X_dev.shape)
print("Y_seg shape:", Y_seg_dev.shape)
print("Y_class shape:", Y_class_dev.shape)
print("X dtype:", X_dev.dtype)
print("Y_seg dtype:", Y_seg_dev.dtype)
print("Y_class dtype:", Y_class_dev.dtype)

In [ ]:
# CELL 6 — VALIDATE PATCH DATA

assert X_dev.ndim == 4
assert X_dev.shape[1:] == (64, 64, 64)

assert Y_seg_dev.ndim in (4, 5)
if Y_seg_dev.ndim == 4:
    assert Y_seg_dev.shape[1:] == (64, 64, 64)
else:
    assert Y_seg_dev.shape[1:] == (64, 64, 64, 1)

assert Y_class_dev.ndim in (1, 2)
assert len(X_dev) == len(Y_seg_dev) == len(Y_class_dev)

class_values = np.unique(np.asarray(Y_class_dev).reshape(-1))
assert np.all(np.isin(class_values, [0, 1, 2, 3, 4])), class_values

print("Patch-data validation: PASS")
print("Number of patches:", len(X_dev))
print("Classification values:", class_values)

## 2. Model configuration

Baseline architecture:

- Input: `64 × 64 × 64 × 1`
- Encoder depth: 4
- Base filters: 16
- Encoder filters: 16, 32, 64, 128
- Bottleneck: 256
- Residual 3D convolution blocks
- U-Net skip connections
- 3D transposed-convolution upsampling
- Segmentation: sigmoid
- Classification: five-class softmax

In [ ]:
# CELL 7 — MODEL CONFIGURATION

INPUT_SHAPE = (64, 64, 64, 1)
PATCH_SIZE = (64, 64, 64)

NUM_CLASSES = 5

CLASS_NAMES = {
    0: "background",
    1: "displaced",
    2: "non-displaced",
    3: "buckle",
    4: "segmental",
}

BASE_FILTERS = 16
DEPTH = 4
DROPOUT_RATE = 0.20

LEARNING_RATE = 1e-4
SEGMENTATION_LOSS_WEIGHT = 1.0
CLASSIFICATION_LOSS_WEIGHT = 1.0

MODEL_NAME = "Enhanced_3D_ResUNet_MultiTask"

print("Input shape:", INPUT_SHAPE)
print("Number of classes:", NUM_CLASSES)
print("Class names:", CLASS_NAMES)
print("Base filters:", BASE_FILTERS)
print("Depth:", DEPTH)
print("Dropout:", DROPOUT_RATE)

## 3. Enhanced residual 3D building blocks

In [ ]:
# CELL 8 — RESIDUAL 3D CONVOLUTION BLOCK

def residual_block_3d(x, filters, name, dropout_rate=0.0):
    shortcut = x

    x = layers.Conv3D(
        filters, 3, padding="same", use_bias=False,
        kernel_initializer="he_normal", name=f"{name}_conv1"
    )(x)
    x = layers.BatchNormalization(name=f"{name}_bn1")(x)
    x = layers.Activation("relu", name=f"{name}_relu1")(x)

    x = layers.Conv3D(
        filters, 3, padding="same", use_bias=False,
        kernel_initializer="he_normal", name=f"{name}_conv2"
    )(x)
    x = layers.BatchNormalization(name=f"{name}_bn2")(x)

    input_channels = shortcut.shape[-1]

    if input_channels != filters:
        shortcut = layers.Conv3D(
            filters, 1, padding="same", use_bias=False,
            kernel_initializer="he_normal",
            name=f"{name}_projection"
        )(shortcut)
        shortcut = layers.BatchNormalization(
            name=f"{name}_projection_bn"
        )(shortcut)

    x = layers.Add(name=f"{name}_add")([x, shortcut])
    x = layers.Activation("relu", name=f"{name}_out")(x)

    if dropout_rate > 0:
        x = layers.Dropout(
            dropout_rate, name=f"{name}_dropout"
        )(x)

    return x

In [ ]:
# CELL 9 — ENCODER DOWNSAMPLING BLOCK

def downsample_block_3d(x, filters, name):
    return layers.Conv3D(
        filters,
        kernel_size=2,
        strides=2,
        padding="same",
        use_bias=False,
        kernel_initializer="he_normal",
        name=name,
    )(x)

In [ ]:
# CELL 10 — DECODER BLOCK

def decoder_block_3d(
    x, skip, filters, name, dropout_rate=0.0
):
    x = layers.Conv3DTranspose(
        filters,
        kernel_size=2,
        strides=2,
        padding="same",
        use_bias=False,
        kernel_initializer="he_normal",
        name=f"{name}_up",
    )(x)

    x = layers.Concatenate(
        name=f"{name}_concat"
    )([x, skip])

    x = residual_block_3d(
        x,
        filters,
        name=f"{name}_res1",
        dropout_rate=dropout_rate,
    )

    return x

## 4. Build the complete multi-task Enhanced 3D ResUNet

The model has a shared residual 3D encoder and two task-specific branches.

**Segmentation branch:** U-Net decoder with residual blocks and skip connections.

**Classification branch:** global feature aggregation followed by dense layers and a five-class softmax output.

In [ ]:
# CELL 11 — BUILD ENHANCED 3D RESUNET

def build_enhanced_3d_resunet(
    input_shape=INPUT_SHAPE,
    num_classes=NUM_CLASSES,
    base_filters=BASE_FILTERS,
    dropout_rate=DROPOUT_RATE,
):
    inputs = keras.Input(
        shape=input_shape,
        name="ct_input",
    )

    # Encoder
    e1 = residual_block_3d(
        inputs, base_filters, "enc1", dropout_rate
    )
    d1 = downsample_block_3d(
        e1, base_filters * 2, "down1"
    )

    e2 = residual_block_3d(
        d1, base_filters * 2, "enc2", dropout_rate
    )
    d2 = downsample_block_3d(
        e2, base_filters * 4, "down2"
    )

    e3 = residual_block_3d(
        d2, base_filters * 4, "enc3", dropout_rate
    )
    d3 = downsample_block_3d(
        e3, base_filters * 8, "down3"
    )

    e4 = residual_block_3d(
        d3, base_filters * 8, "enc4", dropout_rate
    )
    d4 = downsample_block_3d(
        e4, base_filters * 16, "down4"
    )

    # Bottleneck
    bottleneck = residual_block_3d(
        d4, base_filters * 16, "bottleneck", dropout_rate
    )

    # Segmentation decoder
    dec4 = decoder_block_3d(
        bottleneck, e4, base_filters * 8, "dec4", dropout_rate
    )
    dec3 = decoder_block_3d(
        dec4, e3, base_filters * 4, "dec3", dropout_rate
    )
    dec2 = decoder_block_3d(
        dec3, e2, base_filters * 2, "dec2", dropout_rate
    )
    dec1 = decoder_block_3d(
        dec2, e1, base_filters, "dec1", dropout_rate
    )

    segmentation_output = layers.Conv3D(
        1,
        kernel_size=1,
        activation="sigmoid",
        name="segmentation_output",
    )(dec1)

    # Classification branch
    cls = layers.GlobalAveragePooling3D(
        name="classification_global_pool"
    )(bottleneck)

    cls = layers.Dense(
        base_filters * 8,
        activation="relu",
        kernel_initializer="he_normal",
        name="classification_dense1",
    )(cls)

    cls = layers.Dropout(
        dropout_rate,
        name="classification_dropout",
    )(cls)

    cls = layers.Dense(
        base_filters * 4,
        activation="relu",
        kernel_initializer="he_normal",
        name="classification_dense2",
    )(cls)

    classification_output = layers.Dense(
        num_classes,
        activation="softmax",
        name="classification_output",
    )(cls)

    return keras.Model(
        inputs=inputs,
        outputs={
            "segmentation_output": segmentation_output,
            "classification_output": classification_output,
        },
        name=MODEL_NAME,
    )

In [ ]:
# CELL 12 — INSTANTIATE MODEL

model = build_enhanced_3d_resunet()
model.summary()

print("Model construction: PASS")

In [ ]:
# CELL 13 — SHAPE VALIDATION

assert model.input_shape == (None, 64, 64, 64, 1)

assert (
    model.output_shape["segmentation_output"]
    == (None, 64, 64, 64, 1)
)

assert (
    model.output_shape["classification_output"]
    == (None, NUM_CLASSES)
)

print("Input shape:", model.input_shape)
print("Segmentation output:",
      model.output_shape["segmentation_output"])
print("Classification output:",
      model.output_shape["classification_output"])
print("Shape validation: PASS")

In [ ]:
# CELL 14 — FORWARD PASS TEST

test_batch_size = min(2, len(X_dev))

x_test = np.asarray(
    X_dev[:test_batch_size],
    dtype=np.float32
)

x_test = np.expand_dims(x_test, axis=-1)

outputs = model(x_test, training=False)

seg_pred = outputs["segmentation_output"].numpy()
cls_pred = outputs["classification_output"].numpy()

assert seg_pred.shape == (
    test_batch_size, 64, 64, 64, 1
)

assert cls_pred.shape == (
    test_batch_size, NUM_CLASSES
)

assert np.all(seg_pred >= 0.0)
assert np.all(seg_pred <= 1.0)

cls_sums = cls_pred.sum(axis=1)
assert np.allclose(cls_sums, 1.0, atol=1e-5)

print("Forward pass: PASS")
print("Segmentation prediction shape:", seg_pred.shape)
print("Classification prediction shape:", cls_pred.shape)
print("Classification probability sums:", cls_sums)

## 5. Compile the multi-task model

Baseline losses:

- Segmentation: Binary Cross-Entropy
- Classification: Sparse Categorical Cross-Entropy

Baseline loss weights:

- Segmentation = 1.0
- Classification = 1.0

Compilation here is only for validation. Actual optimization, callbacks, augmentation, and training belong to Notebook 05.

In [ ]:
# CELL 15 — BASELINE MULTI-TASK COMPILATION

model.compile(
    optimizer=keras.optimizers.Adam(
        learning_rate=LEARNING_RATE
    ),
    loss={
        "segmentation_output":
            keras.losses.BinaryCrossentropy(),
        "classification_output":
            keras.losses.SparseCategoricalCrossentropy(),
    },
    loss_weights={
        "segmentation_output":
            SEGMENTATION_LOSS_WEIGHT,
        "classification_output":
            CLASSIFICATION_LOSS_WEIGHT,
    },
    metrics={
        "segmentation_output": [
            keras.metrics.BinaryAccuracy(
                name="seg_accuracy"
            )
        ],
        "classification_output": [
            keras.metrics.SparseCategoricalAccuracy(
                name="cls_accuracy"
            )
        ],
    },
)

print("Model compilation: PASS")

In [ ]:
# CELL 16 — LOSS AND GRADIENT SMOKE TEST

y_seg_test = np.asarray(
    Y_seg_dev[:test_batch_size],
    dtype=np.float32
)

if y_seg_test.ndim == 4:
    y_seg_test = np.expand_dims(
        y_seg_test, axis=-1
    )

y_cls_test = np.asarray(
    Y_class_dev[:test_batch_size],
    dtype=np.int32
).reshape(-1)

with tf.GradientTape() as tape:
    pred = model(x_test, training=True)

    total_loss = model.compiled_loss(
        {
            "segmentation_output": y_seg_test,
            "classification_output": y_cls_test,
        },
        pred,
        regularization_losses=model.losses,
    )

gradients = tape.gradient(
    total_loss,
    model.trainable_variables
)

valid_gradients = [
    g for g in gradients if g is not None
]

assert len(valid_gradients) == len(
    model.trainable_variables
)

assert all(
    bool(
        tf.reduce_all(
            tf.math.is_finite(g)
        ).numpy()
    )
    for g in valid_gradients
)

assert np.isfinite(float(total_loss.numpy()))

print("Loss/gradient smoke test: PASS")
print("Total loss:", float(total_loss.numpy()))
print("Trainable variables:",
      len(model.trainable_variables))

## 6. Save persistent model artifacts

Notebook 05 will load these files from the published Kaggle Dataset rather than depending on Notebook 04's temporary runtime.

In [ ]:
# CELL 17 — CREATE PERSISTENT OUTPUT DIRECTORY

MODEL_ARTIFACT_ROOT = Path(
    "/kaggle/working/model_artifacts"
)

MODEL_ARTIFACT_ROOT.mkdir(
    parents=True,
    exist_ok=True
)

MODEL_PATH = (
    MODEL_ARTIFACT_ROOT
    / "enhanced_3d_resunet_multitask.keras"
)

CONFIG_PATH = (
    MODEL_ARTIFACT_ROOT
    / "model_config.json"
)

METADATA_PATH = (
    MODEL_ARTIFACT_ROOT
    / "dataset-metadata.json"
)

print("Model artifact directory:",
      MODEL_ARTIFACT_ROOT)

In [ ]:
# CELL 18 — SAVE MODEL

model.save(MODEL_PATH)

assert MODEL_PATH.exists()
assert MODEL_PATH.stat().st_size > 0

print("Model saved:", MODEL_PATH)
print(
    "File size (MB):",
    round(MODEL_PATH.stat().st_size / (1024**2), 2)
)

In [ ]:
# CELL 19 — SAVE MODEL CONFIGURATION

model_config = {
    "model_name": MODEL_NAME,
    "input_shape": list(INPUT_SHAPE),
    "patch_size": list(PATCH_SIZE),
    "num_classes": NUM_CLASSES,
    "class_names": CLASS_NAMES,
    "base_filters": BASE_FILTERS,
    "depth": DEPTH,
    "dropout_rate": DROPOUT_RATE,

    "segmentation_output": {
        "channels": 1,
        "activation": "sigmoid",
        "task": "binary fracture segmentation",
    },

    "classification_output": {
        "classes": NUM_CLASSES,
        "activation": "softmax",
        "task": "five-class patch classification",
    },

    "optimizer": "Adam",
    "learning_rate": LEARNING_RATE,

    "losses": {
        "segmentation_output":
            "BinaryCrossentropy",
        "classification_output":
            "SparseCategoricalCrossentropy",
    },

    "loss_weights": {
        "segmentation_output":
            SEGMENTATION_LOSS_WEIGHT,
        "classification_output":
            CLASSIFICATION_LOSS_WEIGHT,
    },

    "seed": SEED,
    "source_notebook": "Notebook 04",
    "purpose": (
        "Persistent architecture artifact for Notebook 05"
    ),
}

with open(
    CONFIG_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        model_config,
        f,
        indent=2
    )

assert CONFIG_PATH.exists()

print("Configuration saved:", CONFIG_PATH)

In [ ]:
# CELL 20 — CREATE KAGGLE DATASET METADATA

dataset_metadata = {
    "title":
        "RibFrac Enhanced 3D ResUNet Model Artifacts",
    "id":
        "waboke/ribfrac-model-artifacts",
    "licenses": [
        {"name": "CC-BY-4.0"}
    ],
    "description": (
        "Persistent Enhanced 3D ResUNet multi-task "
        "model artifacts produced by Notebook 04 "
        "for RibFrac rib-fracture segmentation and "
        "five-class classification. Contains the "
        "validated Keras model and reproducible "
        "model configuration for Notebook 05."
    ),
}

with open(
    METADATA_PATH,
    "w",
    encoding="utf-8"
) as f:
    json.dump(
        dataset_metadata,
        f,
        indent=2
    )

assert METADATA_PATH.exists()

print("Kaggle metadata saved:", METADATA_PATH)

## 7. Reload and validate the persisted model

This step is important: Notebook 05 must be able to load the saved `.keras` artifact independently.

In [ ]:
# CELL 21 — RELOAD SAVED MODEL

reloaded_model = keras.models.load_model(
    MODEL_PATH,
    compile=False
)

assert reloaded_model.input_shape == model.input_shape
assert reloaded_model.output_shape == model.output_shape

reloaded_outputs = reloaded_model(
    x_test,
    training=False
)

reloaded_seg = (
    reloaded_outputs["segmentation_output"]
    .numpy()
)

reloaded_cls = (
    reloaded_outputs["classification_output"]
    .numpy()
)

assert reloaded_seg.shape == seg_pred.shape
assert reloaded_cls.shape == cls_pred.shape

assert np.allclose(
    reloaded_cls.sum(axis=1),
    1.0,
    atol=1e-5
)

print("Reload validation: PASS")
print("Reloaded model:", reloaded_model.name)

In [ ]:
# CELL 22 — FINAL ARTIFACT VALIDATION

required_artifacts = [
    MODEL_PATH,
    CONFIG_PATH,
    METADATA_PATH,
]

for path in required_artifacts:
    assert path.exists(), f"Missing artifact: {path}"
    assert path.stat().st_size > 0, f"Empty artifact: {path}"

with open(
    CONFIG_PATH,
    "r",
    encoding="utf-8"
) as f:
    saved_config = json.load(f)

assert saved_config["model_name"] == MODEL_NAME
assert saved_config["input_shape"] == list(INPUT_SHAPE)
assert saved_config["patch_size"] == list(PATCH_SIZE)
assert saved_config["num_classes"] == NUM_CLASSES
assert saved_config["base_filters"] == BASE_FILTERS
assert saved_config["depth"] == DEPTH

print("=" * 60)
print("NOTEBOOK 04 COMPLETION: PASS")
print("=" * 60)
print("Model:", MODEL_PATH)
print("Configuration:", CONFIG_PATH)
print("Metadata:", METADATA_PATH)
print()
print("Publish as:")
print("waboke/ribfrac-model-artifacts")
print()
print("Notebook 05 will load:")
print("  enhanced_3d_resunet_multitask.keras")
print("  model_config.json")

## 8. Publish the model artifacts as a persistent Kaggle Dataset

After all validation cells pass, publish the directory.

### First-time creation

```python
!kaggle datasets create -p /kaggle/working/model_artifacts
```

### Updating an existing dataset

```python
!kaggle datasets version -p /kaggle/working/model_artifacts -m "Updated Notebook 04 model artifacts"
```

Target dataset:

```text
waboke/ribfrac-model-artifacts
```

If Kaggle reports that the dataset already exists, use the `version` command instead of `create`.

In [ ]:
# CELL 23 — OPTIONAL KAGGLE PUBLISH COMMAND

# Run ONE of these when ready:
#
# First publication:
# !kaggle datasets create -p /kaggle/working/model_artifacts
#
# Existing dataset update:
# !kaggle datasets version -p /kaggle/working/model_artifacts -m "Updated Notebook 04 model artifacts"

print("Publishing is optional.")
print("Use the commands in the preceding markdown cell.")

# Notebook 04 completion criteria

The notebook is complete when:

- Notebook 03 patch data is located.
- Patch shapes and class labels are valid.
- Enhanced 3D ResUNet is constructed.
- Input shape is `64×64×64×1`.
- Segmentation output is `64×64×64×1`.
- Classification output is five-class softmax.
- Forward pass succeeds.
- Classification probabilities sum to one.
- Multi-task losses can be calculated.
- Gradients are finite.
- The model saves successfully.
- `model_config.json` is saved.
- `dataset-metadata.json` is saved.
- The saved model reloads successfully.
- Final artifact validation passes.

### Expected output

```text
model_artifacts/
├── enhanced_3d_resunet_multitask.keras
├── model_config.json
└── dataset-metadata.json
```

After publishing this directory as `waboke/ribfrac-model-artifacts`, Notebook 05 can load the exact validated architecture independently of Notebook 04.

**Important:** successful execution of Notebook 04 demonstrates architectural and computational validity; it does not establish predictive performance. Training belongs to Notebook 05 and final evaluation belongs to Notebook 06.